# Regression NBA Model


## Configuration

## Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from nba_ou.data_processing.missing_data.clean_df_for_training import (
    clean_dataframe_for_training,
)
from nba_ou.modeling.modeling import (
    TemporalDecaySampleWeightRegressor,
    evaluate_day_by_day_walk_forward,
    split_latest_dates_holdout,
    make_walk_forward_last_n_seasons_splits,
    validate_time_splits,
    make_test_anchored_walk_forward_splits,
    assert_valid_time_splits,
    save_model_bundle,
    load_model_bundle,
)


In [2]:
TARGET_COL = "LINE_ERROR"
SAMPLE_WEIGHT_LAMBDA = 0.005
SAMPLE_WEIGHT_LAMBDA_BOUNDS = (1e-4, 0.01)
TRAIN_GAMES = 6750

## Load Data

In [3]:
nan_threshold = 5
max_na_per_row = 3


data_path = "/home/adrian_alvarez/Projects/NBA_over_under_predictor/data/train_data/"
name = "all_odds_training_data_until_20260405.csv"

path = data_path + name

header_cols = pd.read_csv(path, nrows=0).columns
dtype_dict = {col: str for col in header_cols if "ID" in col.upper()}

df_stats = pd.read_csv(
    path,
    dtype=dtype_dict,
)
df_stats["GAME_DATE"] = pd.to_datetime(df_stats["GAME_DATE"]).dt.strftime("%Y-%m-%d")


In [4]:
exclude = "fanatics_sportsbook"

In [5]:
df_to_train = clean_dataframe_for_training(df_stats, nan_threshold=nan_threshold, max_na_per_row=max_na_per_row, create_missing_flags=False, verbose=1, keep_columns=['GAME_DATE'], exclude_cols_containing=[exclude])

STARTING DATAFRAME CLEANING PIPELINE
Starting basic cleaning with 11394 rows
Basic cleaning complete: 8785 rows remaining

Starting advanced column cleaning with 2948 columns

Advanced column cleaning complete: 2948 → 1369 columns (1579 removed)


Applying missing data policy...

Missing Data Policy Report:
  Rows dropped: 1 (0.01%)
  Critical columns requiring data: 4
  Columns zero-filled: 104
  Infer pairs applied: 20/136
  Remaining NaN cells: 133425

Dropping rows with more than 3 NaN values...
Removed 1508 rows exceeding NaN threshold
CLEANING COMPLETE
Final shape: (7276, 1369)


In [6]:
import time
time.sleep(5.5*3600)

In [7]:
# Count NAs per column
na_counts = df_to_train.isna().sum()

# Get most common SEASON_YEAR for nulls in each column
most_common_season = []
for col in df_to_train.columns:
    if na_counts[col] > 0:
        null_rows = df_to_train[df_to_train[col].isna()]
        if len(null_rows) > 0 and "SEASON_YEAR" in df_to_train.columns:
            common_season = null_rows["SEASON_YEAR"].mode()
            most_common_season.append(
                common_season.iloc[0] if len(common_season) > 0 else None
            )
        else:
            most_common_season.append(None)
    else:
        most_common_season.append(None)

na_counts_df = pd.DataFrame(
    {
        "Column": na_counts.index,
        "NA_Count": na_counts.values,
        "NA_Percentage": (na_counts.values / len(df_to_train) * 100).round(2),
        "Most_Common_Season_Year": most_common_season,
    }
).sort_values("NA_Count", ascending=False)

na_counts_df[na_counts_df["NA_Count"] > 0]

,Column,NA_Count,NA_Percentage,Most_Common_Season_Year
1167,LEAGUE_GAMES_LAST_1D_BEFORE,266,3.66,2023.0
1359,TRAVEL_RECENCY_RATIO_AWAY_2D_OVER_14D_BEFORE,79,1.09,2019.0
210,ml_betmgm_price_LAST_ALL_5_MATCHES_BEFORE_TEAM...,62,0.85,2019.0
629,ml_betmgm_price_LAST_ALL_5_MATCHES_BEFORE_TEAM...,60,0.82,2019.0
1358,TRAVEL_RECENCY_RATIO_HOME_2D_OVER_14D_BEFORE,41,0.56,2019.0
895,DIFF_FROM_LINE_caesars_LAST_ALL_1_MATCHES_DIFF...,28,0.38,2023.0
538,DIFF_FROM_LINE_caesars_LAST_ALL_1_MATCHES_BEFO...,22,0.30,2023.0
121,DIFF_FROM_LINE_caesars_LAST_ALL_1_MATCHES_BEFO...,15,0.21,2023.0
902,DIFF_FROM_LINE_draftkings_LAST_ALL_1_MATCHES_D...,8,0.11,2020.0
1207,ml_consensus_opener_price_away,8,0.11,2025.0


In [8]:
BET365_LINE_COL = "TOTAL_LINE_bet365"
# BET365_LINE_COL = "total_bet365_line_over"

# Ensure the main scoring line and actual total exist.
df_to_train = df_to_train.dropna(subset=[BET365_LINE_COL, "TOTAL_POINTS"]).copy()

In [9]:
df_to_train["LINE_ERROR"] = df_to_train["TOTAL_POINTS"] - df_to_train[BET365_LINE_COL]


In [10]:
df_to_train["GAME_DATE"] = pd.to_datetime(df_to_train["GAME_DATE"])
df_to_train = df_to_train.sort_values("GAME_DATE").reset_index(drop=True)

# Count games per season
games_per_season = df_to_train.groupby("SEASON_YEAR").size()
print(games_per_season)


SEASON_YEAR
2019     296
2020    1071
2021    1221
2022    1202
2023    1169
2024    1232
2025    1085
dtype: int64


## Train / Test

In [11]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    root_mean_squared_error,
)
from sklearn.model_selection import cross_validate
from xgboost import XGBRegressor

from nba_ou.modeling.optuna_error_line import (
    fit_best_xgb_error_line,
    select_best_trial_lexicographic,
    summarize_lexicographic_candidates,
    summarize_optuna_trials,
    tune_xgb_error_line_optuna,
)
from nba_ou.modeling.scorers import (
    OverUnderScorerLineError,
    OverUnderScorerLineErrorMinEdge,
    evaluate_error_thresholds,
    over_under_betting_accuracy_error_line,
    over_under_betting_accuracy_error_line_with_min_edge,
)

In [12]:
df_dev, df_test_final = split_latest_dates_holdout(
    df=df_to_train,
    date_col="GAME_DATE",
    test_size=0.04,
)

print(f"Development set size: {len(df_dev)}")
print(f"Final test set size: {len(df_test_final)}")
print(
    "Final test date range:",
    df_test_final["GAME_DATE"].min(),
    "->",
    df_test_final["GAME_DATE"].max(),
)

Development set size: 6984
Final test set size: 292
Final test date range: 2026-02-26 00:00:00 -> 2026-04-05 00:00:00


In [13]:
def build_recency_sample_weights(df, date_col="GAME_DATE", lambda_=SAMPLE_WEIGHT_LAMBDA):
    dates = pd.to_datetime(df[date_col])
    max_date = dates.max()
    age_days = (max_date - dates).dt.days
    weights = np.exp(-lambda_ * age_days)
    return pd.Series(weights, index=df.index, name="sample_weight")

EXCLUDE_COLS = [
    "TOTAL_POINTS",
    "LINE_ERROR",
    "SEASON_YEAR",
    "GAME_DATE",
]

X_dev = df_dev.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(df_dev[TARGET_COL], errors="coerce")
sample_weight_dev = build_recency_sample_weights(df_dev)

X_test_final = df_test_final.drop(columns=EXCLUDE_COLS, errors="ignore")
y_test_final = pd.to_numeric(df_test_final[TARGET_COL], errors="coerce")

print(f"X_dev shape: {X_dev.shape}")
print(f"X_test_final shape: {X_test_final.shape}")
print(
    f"Recency sample weights lambda={SAMPLE_WEIGHT_LAMBDA}: "
    f"min={sample_weight_dev.min():.4f}, max={sample_weight_dev.max():.4f}"
)


X_dev shape: (6984, 1366)
X_test_final shape: (292, 1366)
Recency sample weights lambda=0.005: min=0.0000, max=1.0000


In [14]:
ou_scorer = OverUnderScorerLineError()
ou_scorer_edge_2 = OverUnderScorerLineErrorMinEdge(min_edge=2)
ou_scorer_edge_4 = OverUnderScorerLineErrorMinEdge(min_edge=4)

scoring = {
    "MSE": "neg_mean_squared_error",
    "RMSE": "neg_root_mean_squared_error",
    "MAE": "neg_mean_absolute_error",
    "R2": "r2",
    "OU_Betting_Accuracy": ou_scorer,
    "OU_Betting_Accuracy_Edge_2": ou_scorer_edge_2,
    "OU_Betting_Accuracy_Edge_4": ou_scorer_edge_4,
}


def print_metrics(cv_results):
    for sc in scoring.keys():
        train_key = f"train_{sc}"
        test_key = f"test_{sc}"

        train_val = cv_results[train_key].mean()
        test_val = cv_results[test_key].mean()

        if sc in {"MSE", "RMSE", "MAE"}:
            train_val = -train_val
            test_val = -test_val

        if sc.startswith("OU_Betting_Accuracy"):
            print(f"Train {sc}: {train_val:.2%}")
            print(f"Validation {sc}: {test_val:.2%}")
        else:
            print(f"Train {sc}: {train_val:.5f}")
            print(f"Validation {sc}: {test_val:.5f}")
        print()


In [15]:
DAY_BY_DAY_METRIC_NAME = "OU_Betting_Accuracy"
DAY_BY_DAY_THRESHOLDS = (1, 2, 3)


def summarize_walk_forward_thresholds(predictions_df, thresholds=DAY_BY_DAY_THRESHOLDS):
    y_true = pd.to_numeric(predictions_df["y_true"], errors="coerce").to_numpy(dtype=float)
    y_pred = pd.to_numeric(predictions_df["y_pred"], errors="coerce").to_numpy(dtype=float)
    margin = np.abs(y_pred)
    n_total = len(predictions_df)

    rows = []
    for t in thresholds:
        mask = margin > t
        n = int(mask.sum())
        acc = (
            np.nan
            if n == 0
            else over_under_betting_accuracy_error_line(
                y_true_error=y_true[mask],
                y_pred_error=y_pred[mask],
            )
        )
        rows.append(
            {
                "threshold_abs_pred_error_gt": t,
                "n_games": n,
                "pct_of_test": (n / n_total) if n_total else np.nan,
                "directional_accuracy": acc,
            }
        )

    return pd.DataFrame(rows)


def run_day_by_day_walk_forward_evaluation(
    *,
    label,
    df_dev,
    df_test_final,
    fit_and_predict,
    max_games=TRAIN_GAMES,
    metric_name=DAY_BY_DAY_METRIC_NAME,
    thresholds=DAY_BY_DAY_THRESHOLDS,
):
    result = evaluate_day_by_day_walk_forward(
        df_dev=df_dev,
        df_test_final=df_test_final,
        fit_and_predict=fit_and_predict,
        metric_fn=lambda y_true, y_pred: over_under_betting_accuracy_error_line(
            y_true_error=y_true,
            y_pred_error=y_pred,
        ),
        target_col=TARGET_COL,
        max_games=max_games,
        metric_name=metric_name,
    )

    threshold_results = summarize_walk_forward_thresholds(
        result.predictions,
        thresholds=thresholds,
    )

    print(f"{label} mean day-by-day {metric_name}: {result.mean_metric:.2%}")
    display(result.daily_results.style.format({metric_name: "{:.2%}"}))
    print(f"{label} thresholded walk-forward accuracy")
    display(
        threshold_results.style.format(
            {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
        )
    )
    return result, threshold_results


In [16]:
splits, fold_info = make_test_anchored_walk_forward_splits(
    df=df_dev,
    date_col="GAME_DATE",
    season_col="SEASON_YEAR",
    test_games=25,
    step_games_between_tests=25,
    train_games=TRAIN_GAMES,
    min_train_games=TRAIN_GAMES*0.8,
    max_folds=15,
    verbose=1,
)

assert_valid_time_splits(df_dev, splits)



Created 15 test-anchored walk-forward folds
 fold  train_n_games  test_n_games train_start_date train_end_date test_start_date test_end_date  test_season
    1           6100            25       2019-11-10     2025-04-11      2025-04-13    2025-04-23         2024
    2           6191            32       2019-11-10     2025-06-22      2025-10-27    2025-11-03         2025
    3           6252            30       2019-11-10     2025-11-07      2025-11-08    2025-11-11         2025
    4           6311            31       2019-11-10     2025-11-15      2025-11-16    2025-11-19         2025
    5           6370            33       2019-11-10     2025-11-23      2025-11-24    2025-11-28         2025
    6           6428            32       2019-11-10     2025-12-01      2025-12-02    2025-12-05         2025
    7           6485            36       2019-11-10     2025-12-11      2025-12-12    2025-12-18         2025
    8           6549            28       2019-11-10     2025-12-22      2025

In [17]:
season_bl = DummyRegressor(strategy="mean")

cv_results = cross_validate(
    season_bl,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("DummyRegressor baseline")
print_metrics(cv_results)


DummyRegressor baseline
Train MSE: 293.84236
Validation MSE: 275.33303

Train RMSE: 17.14183
Validation RMSE: 16.42756

Train MAE: 13.65532
Validation MAE: 13.33285

Train R2: 0.00000
Validation R2: -0.02941

Train OU_Betting_Accuracy: 51.75%
Validation OU_Betting_Accuracy: 50.94%

Train OU_Betting_Accuracy_Edge_2: 0.00%
Validation OU_Betting_Accuracy_Edge_2: 0.00%

Train OU_Betting_Accuracy_Edge_4: 0.00%
Validation OU_Betting_Accuracy_Edge_4: 0.00%



In [18]:
lr = LinearRegression()

cv_results = cross_validate(
    lr,
    X_dev.fillna(0),
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=-1,
)

print("Linear Regression")
print_metrics(cv_results)


Linear Regression
Train MSE: 236.27026
Validation MSE: 3849.37579

Train RMSE: 15.37083
Validation RMSE: 36.88582

Train MAE: 12.17856
Validation MAE: 24.30783

Train R2: 0.19593
Validation R2: -11.97273

Train OU_Betting_Accuracy: 64.31%
Validation OU_Betting_Accuracy: 52.56%

Train OU_Betting_Accuracy_Edge_2: 68.15%
Validation OU_Betting_Accuracy_Edge_2: 52.94%

Train OU_Betting_Accuracy_Edge_4: 72.05%
Validation OU_Betting_Accuracy_Edge_4: 53.84%



In [19]:
xgb_reg_no_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=100,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

cv_results_no_weights = cross_validate(
    xgb_reg_no_weights,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost no sample weights")
print_metrics(cv_results_no_weights)

XGBoost no sample weights
Train MSE: 255.40999
Validation MSE: 281.47530

Train RMSE: 15.98149
Validation RMSE: 16.62445

Train MAE: 12.71995
Validation MAE: 13.51551

Train R2: 0.13079
Validation R2: -0.05743

Train OU_Betting_Accuracy: 67.87%
Validation OU_Betting_Accuracy: 50.17%

Train OU_Betting_Accuracy_Edge_2: 83.60%
Validation OU_Betting_Accuracy_Edge_2: 46.09%

Train OU_Betting_Accuracy_Edge_4: 94.40%
Validation OU_Betting_Accuracy_Edge_4: 38.33%



In [20]:
xgb_reg_no_weights.fit(X_dev, y_dev)

y_pred_test_error = xgb_reg_no_weights.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 306.30317
RMSE: 17.50152
MAE: 13.57130
OU_Betting_Accuracy: 52.45%
OU_Betting_Accuracy_Edge_2: 62.16%
OU_Betting_Accuracy_Edge_4: 50.00%


In [21]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=xgb_reg_no_weights,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,292,100.0%,52.45%
1,1,177,60.6%,51.72%
2,2,75,25.7%,62.16%
3,3,24,8.2%,50.00%
4,4,4,1.4%,50.00%
5,5,0,0.0%,nan%
6,6,0,0.0%,nan%
7,7,0,0.0%,nan%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [22]:
def fit_and_predict_xgb_no_weights_day_by_day(train_df, test_df):
    model = XGBRegressor(**xgb_reg_no_weights.get_params())

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_no_weights, day_by_day_no_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost no sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_no_weights_day_by_day,
    max_games=TRAIN_GAMES,
)

XGBoost no sample weights mean day-by-day OU_Betting_Accuracy: 51.43%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-02-26 00:00:00,6750,10,2020-03-10 00:00:00,2026-02-25 00:00:00,55.56%
1,2026-02-27 00:00:00,6750,5,2020-07-31 00:00:00,2026-02-26 00:00:00,40.00%
2,2026-02-28 00:00:00,6750,5,2020-08-01 00:00:00,2026-02-27 00:00:00,80.00%
3,2026-03-01 00:00:00,6750,11,2020-08-02 00:00:00,2026-02-28 00:00:00,45.45%
4,2026-03-02 00:00:00,6750,4,2020-08-04 00:00:00,2026-03-01 00:00:00,100.00%
5,2026-03-03 00:00:00,6750,10,2020-08-05 00:00:00,2026-03-02 00:00:00,70.00%
6,2026-03-04 00:00:00,6750,5,2020-08-06 00:00:00,2026-03-03 00:00:00,60.00%
7,2026-03-05 00:00:00,6750,9,2020-08-07 00:00:00,2026-03-04 00:00:00,77.78%
8,2026-03-06 00:00:00,6750,7,2020-08-09 00:00:00,2026-03-05 00:00:00,14.29%
9,2026-03-07 00:00:00,6750,5,2021-01-01 00:00:00,2026-03-06 00:00:00,60.00%


XGBoost no sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,169,57.9%,49.70%
1,2,73,25.0%,47.22%
2,3,30,10.3%,53.33%


## Check weighted

In [23]:
xgb_reg_weights = XGBRegressor(
    max_depth=3,
    learning_rate=0.05,
    n_estimators=35,
    subsample=0.65,
    colsample_bytree=0.68,
    reg_alpha=5.28,
    reg_lambda=1.3,
    min_child_weight=5.08,
    gamma=0.0085,
    n_jobs=-1,
    random_state=16,
)

weighted_xgb = TemporalDecaySampleWeightRegressor(
    estimator=xgb_reg_weights,
    dates=df_dev["GAME_DATE"],
    lambda_=SAMPLE_WEIGHT_LAMBDA,
)

cv_results_weights = cross_validate(
    weighted_xgb,
    X_dev,
    y_dev,
    cv=splits,
    scoring=scoring,
    return_train_score=True,
    n_jobs=1,
)

print("XGBoost with sample weights (per-fold decay)")
print_metrics(cv_results_weights)

XGBoost with sample weights (per-fold decay)
Train MSE: 285.74401
Validation MSE: 281.59837

Train RMSE: 16.90393
Validation RMSE: 16.62058

Train MAE: 13.44422
Validation MAE: 13.49473

Train R2: 0.02756
Validation R2: -0.05690

Train OU_Betting_Accuracy: 55.65%
Validation OU_Betting_Accuracy: 48.57%

Train OU_Betting_Accuracy_Edge_2: 63.62%
Validation OU_Betting_Accuracy_Edge_2: 50.89%

Train OU_Betting_Accuracy_Edge_4: 81.21%
Validation OU_Betting_Accuracy_Edge_4: 45.71%



In [24]:
weighted_xgb.fit(X_dev, y_dev)

y_pred_test_error = weighted_xgb.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 309.06753
RMSE: 17.58032
MAE: 13.54261
OU_Betting_Accuracy: 54.20%
OU_Betting_Accuracy_Edge_2: 55.66%
OU_Betting_Accuracy_Edge_4: 66.67%


In [25]:
results_df, y_pred_test_error = evaluate_error_thresholds(
    model=weighted_xgb,
    X_test=X_test_final,
    y_test_error=y_test_final,
    thresholds=range(0, 11),
)

display(
    results_df.style.format(
        {"pct_of_test": "{:.1%}", "directional_accuracy": "{:.2%}"}
    )
)


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,0,292,100.0%,54.20%
1,1,184,63.0%,55.25%
2,2,108,37.0%,55.66%
3,3,53,18.2%,53.85%
4,4,21,7.2%,66.67%
5,5,9,3.1%,66.67%
6,6,2,0.7%,100.00%
7,7,1,0.3%,100.00%
8,8,0,0.0%,nan%
9,9,0,0.0%,nan%


In [26]:
def fit_and_predict_xgb_weights_day_by_day(train_df, test_df):
    base_model = XGBRegressor(**xgb_reg_weights.get_params())
    model = TemporalDecaySampleWeightRegressor(
        estimator=base_model,
        dates=train_df["GAME_DATE"],
        lambda_=SAMPLE_WEIGHT_LAMBDA,
    )

    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model.fit(X_train, y_train)
    return model.predict(X_test)


day_by_day_weights, day_by_day_weights_thresholds = run_day_by_day_walk_forward_evaluation(
    label="XGBoost with sample weights",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_xgb_weights_day_by_day,
    max_games=TRAIN_GAMES,
)


XGBoost with sample weights mean day-by-day OU_Betting_Accuracy: 54.42%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-02-26 00:00:00,6750,10,2020-03-10 00:00:00,2026-02-25 00:00:00,55.56%
1,2026-02-27 00:00:00,6750,5,2020-07-31 00:00:00,2026-02-26 00:00:00,20.00%
2,2026-02-28 00:00:00,6750,5,2020-08-01 00:00:00,2026-02-27 00:00:00,100.00%
3,2026-03-01 00:00:00,6750,11,2020-08-02 00:00:00,2026-02-28 00:00:00,54.55%
4,2026-03-02 00:00:00,6750,4,2020-08-04 00:00:00,2026-03-01 00:00:00,50.00%
5,2026-03-03 00:00:00,6750,10,2020-08-05 00:00:00,2026-03-02 00:00:00,100.00%
6,2026-03-04 00:00:00,6750,5,2020-08-06 00:00:00,2026-03-03 00:00:00,80.00%
7,2026-03-05 00:00:00,6750,9,2020-08-07 00:00:00,2026-03-04 00:00:00,77.78%
8,2026-03-06 00:00:00,6750,7,2020-08-09 00:00:00,2026-03-05 00:00:00,28.57%
9,2026-03-07 00:00:00,6750,5,2021-01-01 00:00:00,2026-03-06 00:00:00,40.00%


XGBoost with sample weights thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,201,68.8%,55.56%
1,2,125,42.8%,57.72%
2,3,59,20.2%,52.63%


# Optuna

In [27]:
study = tune_xgb_error_line_optuna(
    X=X_dev,
    y=y_dev,
    sample_weight_dates=df_dev["GAME_DATE"],
    tune_sample_weight_lambda=True,
    sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    splits=splits,
    n_trials=80,
    timeout=4.5 * 3600,
    # timeout=600,

    objective_name="reg:squarederror",
    study_name="xgb_error_line_mae",
)

best_trial_lexi = select_best_trial_lexicographic(
    study,
    mae_tolerance_abs=0.05,
)

print("Optuna best by MAE only")
print("Trial:", study.best_trial.number)
print("Best CV MAE:", study.best_value)
print("Mean OU accuracy:", study.best_trial.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", study.best_trial.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", study.best_trial.user_attrs.get("mean_ou_acc_edge_4"))

print("\nSelected trial after MAE-first / OU-second ranking")
print("Trial:", best_trial_lexi.number)
print("CV MAE:", best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value))
print("Mean RMSE:", best_trial_lexi.user_attrs.get("mean_rmse"))
print("Mean R2:", best_trial_lexi.user_attrs.get("mean_r2"))
print("Mean OU accuracy:", best_trial_lexi.user_attrs.get("mean_ou_acc"))
print("Mean OU accuracy edge 2:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_2"))
print("Mean OU accuracy edge 4:", best_trial_lexi.user_attrs.get("mean_ou_acc_edge_4"))
print("Median best_iteration:", best_trial_lexi.user_attrs.get("median_best_iteration"))
print("Params:")
for k, v in best_trial_lexi.params.items():
    print(f"{k}: {v}")

trials_df = summarize_optuna_trials(study)
display(
    trials_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

candidates_df = summarize_lexicographic_candidates(
    study,
    mae_tolerance_abs=0.05,
)
display(
    candidates_df.head(15).style.format(
        {
            "value_mae": "{:.4f}",
            "mean_mae": "{:.4f}",
            "mean_rmse": "{:.4f}",
            "mean_r2": "{:.4f}",
            "mean_ou_acc": "{:.2%}",
            "mean_ou_acc_edge_2": "{:.2%}",
            "mean_ou_acc_edge_4": "{:.2%}",
        }
    )
)

[I 2026-04-07 05:30:39,050] A new study created in memory with name: xgb_error_line_mae


  0%|          | 0/80 [00:00<?, ?it/s]

[I 2026-04-07 05:37:31,753] Trial 0 finished with value: 13.252899851791199 and parameters: {'max_depth': 2, 'min_child_weight': 18.346704707583235, 'gamma': 1.6970342240854253, 'subsample': 0.5682407800531226, 'colsample_bytree': 0.5123279759064977, 'learning_rate': 0.011926786034588454, 'reg_alpha': 1.8771791376898666, 'reg_lambda': 1.897469395521307, 'sample_weight_lambda': 0.00013824509574177668}. Best is trial 0 with value: 13.252899851791199.
[I 2026-04-07 05:44:45,176] Trial 1 finished with value: 13.214528054719608 and parameters: {'max_depth': 4, 'min_child_weight': 20.290108931287893, 'gamma': 0.3261777842309625, 'subsample': 0.8390562044499359, 'colsample_bytree': 0.4213034780854249, 'learning_rate': 0.012620826760486504, 'reg_alpha': 0.09307011182812809, 'reg_lambda': 15.258811505244246, 'sample_weight_lambda': 0.000848258413826022}. Best is trial 1 with value: 13.214528054719608.
[I 2026-04-07 05:50:20,584] Trial 2 finished with value: 13.10720009717992 and parameters: {'m

,trial,value_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,57,12.9088,16.0881,0.0101,54.71%,60.50%,0.300280,30.78%,63,14,3,8.697734,1.293657,0.659124,0.508661,0.041251,1.917912,1.464429,0.005957
1,59,12.9145,16.0696,0.0138,55.05%,57.77%,0.498254,43.68%,75,52,4,7.559864,0.979676,0.606160,0.518131,0.029202,1.860591,1.469743,0.009875
2,62,12.9247,16.0593,0.0139,59.12%,60.09%,0.535133,36.64%,82,44,4,8.562418,0.994956,0.623565,0.580213,0.033378,1.927096,1.123476,0.008536
3,54,12.9420,16.1356,0.0047,55.95%,54.26%,0.451023,33.79%,48,12,3,10.070550,1.285510,0.643890,0.432735,0.052443,5.347639,1.220171,0.006530
4,41,12.9517,16.1152,0.0066,57.42%,54.83%,0.485025,47.33%,68,20,3,7.907980,1.424017,0.579493,0.499321,0.054742,3.213564,5.752441,0.005144
5,21,12.9583,16.0854,0.0108,55.28%,45.35%,0.407975,28.72%,44,18,3,6.957362,1.430031,0.691413,0.475883,0.057168,3.125436,5.094275,0.005670
6,61,12.9652,16.0879,0.0114,55.75%,55.35%,0.365480,39.19%,62,34,4,8.457753,0.985183,0.602937,0.578654,0.034868,2.107552,1.824090,0.006532
7,25,13.0001,16.0583,0.0160,56.14%,56.29%,0.456204,38.07%,56,30,3,10.610574,2.971392,0.611601,0.428051,0.045642,2.631293,4.285584,0.006780
8,35,13.0009,16.1356,0.0050,55.10%,49.39%,0.501872,43.11%,66,33,2,9.644752,1.597330,0.555929,0.429657,0.054281,1.400550,2.105543,0.004999
9,53,13.0021,16.2185,-0.0049,56.22%,52.66%,0.464921,28.12%,39,18,3,9.954470,1.325066,0.710243,0.436269,0.050975,3.150462,4.243748,0.004563


,trial,value_mae,mean_mae,mean_rmse,mean_r2,mean_ou_acc,mean_ou_acc_edge_2,mean_ou_acc_edge_3,mean_ou_acc_edge_4,mean_best_iteration,median_best_iteration,mae_cutoff,max_depth,min_child_weight,gamma,subsample,colsample_bytree,learning_rate,reg_alpha,reg_lambda,sample_weight_lambda
0,62,12.9247,12.9247,16.0593,0.0139,59.12%,60.09%,0.535133,36.64%,82,44,12.958814,4,8.562418,0.994956,0.623565,0.580213,0.033378,1.927096,1.123476,0.008536
1,41,12.9517,12.9517,16.1152,0.0066,57.42%,54.83%,0.485025,47.33%,68,20,12.958814,3,7.907980,1.424017,0.579493,0.499321,0.054742,3.213564,5.752441,0.005144
2,54,12.9420,12.9420,16.1356,0.0047,55.95%,54.26%,0.451023,33.79%,48,12,12.958814,3,10.070550,1.285510,0.643890,0.432735,0.052443,5.347639,1.220171,0.006530
3,21,12.9583,12.9583,16.0854,0.0108,55.28%,45.35%,0.407975,28.72%,44,18,12.958814,3,6.957362,1.430031,0.691413,0.475883,0.057168,3.125436,5.094275,0.005670
4,59,12.9145,12.9145,16.0696,0.0138,55.05%,57.77%,0.498254,43.68%,75,52,12.958814,4,7.559864,0.979676,0.606160,0.518131,0.029202,1.860591,1.469743,0.009875
5,57,12.9088,12.9088,16.0881,0.0101,54.71%,60.50%,0.300280,30.78%,63,14,12.958814,3,8.697734,1.293657,0.659124,0.508661,0.041251,1.917912,1.464429,0.005957


In [28]:
def fit_and_predict_optuna_day_by_day(train_df, test_df):
    X_train = train_df.drop(columns=EXCLUDE_COLS, errors="ignore")
    y_train = pd.to_numeric(train_df[TARGET_COL], errors="coerce")
    X_test = test_df.drop(columns=EXCLUDE_COLS, errors="ignore")

    model = fit_best_xgb_error_line(
        X_dev=X_train,
        y_dev=y_train,
        sample_weight_dates=train_df["GAME_DATE"],
        sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
        trial=best_trial_lexi,
        objective_name="reg:squarederror",
    )
    return model.predict(X_test)

day_by_day_optuna, day_by_day_optuna_thresholds = run_day_by_day_walk_forward_evaluation(
    label="Optuna-selected XGBoost",
    df_dev=df_dev,
    df_test_final=df_test_final,
    fit_and_predict=fit_and_predict_optuna_day_by_day,
    max_games=TRAIN_GAMES,
)

Optuna-selected XGBoost mean day-by-day OU_Betting_Accuracy: 53.90%


,date,train_n_games,test_n_games,train_start_date,train_end_date,OU_Betting_Accuracy
0,2026-02-26 00:00:00,6750,10,2020-03-10 00:00:00,2026-02-25 00:00:00,55.56%
1,2026-02-27 00:00:00,6750,5,2020-07-31 00:00:00,2026-02-26 00:00:00,40.00%
2,2026-02-28 00:00:00,6750,5,2020-08-01 00:00:00,2026-02-27 00:00:00,80.00%
3,2026-03-01 00:00:00,6750,11,2020-08-02 00:00:00,2026-02-28 00:00:00,63.64%
4,2026-03-02 00:00:00,6750,4,2020-08-04 00:00:00,2026-03-01 00:00:00,50.00%
5,2026-03-03 00:00:00,6750,10,2020-08-05 00:00:00,2026-03-02 00:00:00,90.00%
6,2026-03-04 00:00:00,6750,5,2020-08-06 00:00:00,2026-03-03 00:00:00,80.00%
7,2026-03-05 00:00:00,6750,9,2020-08-07 00:00:00,2026-03-04 00:00:00,88.89%
8,2026-03-06 00:00:00,6750,7,2020-08-09 00:00:00,2026-03-05 00:00:00,42.86%
9,2026-03-07 00:00:00,6750,5,2021-01-01 00:00:00,2026-03-06 00:00:00,40.00%


Optuna-selected XGBoost thresholded walk-forward accuracy


,threshold_abs_pred_error_gt,n_games,pct_of_test,directional_accuracy
0,1,219,75.0%,55.61%
1,2,140,47.9%,56.62%
2,3,84,28.8%,61.45%


In [29]:
total_df = df_dev.tail(TRAIN_GAMES)

In [30]:
X_dev = total_df.drop(columns=EXCLUDE_COLS, errors="ignore")
y_dev = pd.to_numeric(total_df[TARGET_COL], errors="coerce")
sample_weight_dates_dev = total_df["GAME_DATE"]


In [31]:
best_model = fit_best_xgb_error_line(
    X_dev=X_dev,
    y_dev=y_dev,
    sample_weight_dates=sample_weight_dates_dev,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

y_pred_test_error = best_model.predict(X_test_final)

mse = mean_squared_error(y_test_final, y_pred_test_error)
rmse = root_mean_squared_error(y_test_final, y_pred_test_error)
mae = mean_absolute_error(y_test_final, y_pred_test_error)
ou_acc = over_under_betting_accuracy_error_line(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
)
ou_acc_edge_2 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=2,
)
ou_acc_edge_4 = over_under_betting_accuracy_error_line_with_min_edge(
    y_true_error=y_test_final,
    y_pred_error=y_pred_test_error,
    min_edge=4,
)

print("Final test metrics")
print(f"MSE: {mse:.5f}")
print(f"RMSE: {rmse:.5f}")
print(f"MAE: {mae:.5f}")
print(f"OU_Betting_Accuracy: {ou_acc:.2%}")
print(f"OU_Betting_Accuracy_Edge_2: {ou_acc_edge_2:.2%}")
print(f"OU_Betting_Accuracy_Edge_4: {ou_acc_edge_4:.2%}")

Final test metrics
MSE: 308.80018
RMSE: 17.57271
MAE: 13.48792
OU_Betting_Accuracy: 54.55%
OU_Betting_Accuracy_Edge_2: 56.77%
OU_Betting_Accuracy_Edge_4: 63.79%


In [32]:
from nba_ou.modeling.modeling import ModelBundleMetadata, ModelInfo, TrainingMetrics

df_to_train_split_rows = df_to_train.copy()
df_to_train_split_rows = df_to_train_split_rows.tail(TRAIN_GAMES)

X_full = df_to_train_split_rows.drop(columns=EXCLUDE_COLS, errors="ignore")
y_full = pd.to_numeric(df_to_train_split_rows[TARGET_COL], errors="coerce")
sample_weight_dates_full = df_to_train_split_rows["GAME_DATE"]

production_model = fit_best_xgb_error_line(
    X_dev=X_full,
    y_dev=y_full,
    sample_weight_dates=sample_weight_dates_full,
    sample_weight_lambda=best_trial_lexi.params.get("sample_weight_lambda"),
    trial=best_trial_lexi,
    objective_name="reg:squarederror",
)

latest_training_date = pd.to_datetime(df_to_train_split_rows["GAME_DATE"]).max()
model_version = latest_training_date.strftime("%d_%m_%y")
model_name = f"all_seasons_xgb_line_error_{model_version}"

metadata = ModelBundleMetadata(
    model_info=ModelInfo(
        name=model_name,
        model_version=model_version,
        model_type="all_seasons_line_error",
        prediction_source="all_seasons_xgb_line_error",
        training_code_tag="1.0",
    ),
    training_metrics=TrainingMetrics(
        best_params=best_trial_lexi.params,
        selected_trial_number=best_trial_lexi.number,
        mean_best_iteration=best_trial_lexi.user_attrs.get("mean_best_iteration"),
        median_best_iteration=best_trial_lexi.user_attrs.get("median_best_iteration"),
        cv_mae=float(best_trial_lexi.user_attrs.get("mean_mae", best_trial_lexi.value)),
        cv_rmse=best_trial_lexi.user_attrs.get("mean_rmse"),
        cv_ou_acc=best_trial_lexi.user_attrs.get("mean_ou_acc"),
        final_test_mae=float(mae),
        final_test_rmse=float(rmse),
        final_test_ou_acc=float(ou_acc),
        nan_threshold=nan_threshold,
        max_na_per_row=max_na_per_row,
        train_date_min=df_to_train_split_rows["GAME_DATE"].min().to_pydatetime(),
        train_date_max=df_to_train_split_rows["GAME_DATE"].max().to_pydatetime(),
        train_games= TRAIN_GAMES,
        sample_weight_lambda_bounds=SAMPLE_WEIGHT_LAMBDA_BOUNDS,
    ),
)

model_path, meta_path = save_model_bundle(
    model=production_model,
    feature_names=list(X_full.columns),
    out_dir="/home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/all_seasons/",
    metadata=metadata,
)

print(
    f"Production model trained on {len(X_full)} rows using fixed n_estimators from median_best_iteration."
)
print("Saved model :", model_path)
print("Saved metadata:", meta_path)

Production model trained on 6750 rows using fixed n_estimators from median_best_iteration.
Saved model : /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/all_seasons/all_seasons_xgb_line_error_05_04_26.json
Saved metadata: /home/adrian_alvarez/Projects/NBA_over_under_predictor/models/line_error/all_seasons/all_seasons_xgb_line_error_05_04_26.meta.json


In [33]:
best_trial_lexi.user_attrs.get("median_best_iteration")


44